# 04 Classification Margin と Argmax Stability

03 までに、TT/MPS による第1 Linear の weight approximation error が

$$
\Delta W_1\rightarrow\Delta Z_1\rightarrow\Delta H\rightarrow\Delta L
$$

と logits error まで伝播することを確認しました。

今回は **1 sample だけ**を対象に、

$$
\boxed{\|\delta l_n\|_\infty<\frac{\gamma_n}{2}}
$$

なら

$$
\boxed{\operatorname*{argmax}_k(\widetilde l_n)_k
=\operatorname*{argmax}_k(l_n)_k}
$$

となる理由を、理論 → TODO 実装 → 数値検証の順に学びます。

今回はまだ、batch 全体、$\|\Delta L\|_F$ との接続、accuracy 保証、Fashion-MNIST、TT-rank sweep、fine-tuning には進みません。


## 1. 実行環境

初出変数は以下の6項目で説明します。

1. 何を意味するか
2. 数式上の定義・対応
3. shape
4. 各軸の意味
5. 何から計算されるか
6. 何のために使うか

| 変数 | 1. 意味 | 2. 数式対応 | 3. shape | 4. 軸 | 5. 計算元 | 6. 用途 |
|---|---|---|---|---|---|---|
| `dtype` | Tensor の数値型 | なし | setting | なし | `torch.float64` | 丸め誤差を抑える |
| `device` | 計算デバイス | なし | setting | なし | CPU | toy 実験を固定 |
| `rtol` | 相対許容誤差 | なし | scalar | なし | 手動設定 | `assert_close` |
| `atol` | 絶対許容誤差 | なし | scalar | なし | 手動設定 | `assert_close` |


In [25]:
import torch
from torch.testing import assert_close

torch.manual_seed(0)
dtype = torch.float64
device = torch.device("cpu")
rtol = 1e-10
atol = 1e-12


## 2. 1 sample の dense logits

class 数を $c=5$ とし、1 sample の dense logits

$$
l_n\in\mathbb R^c
$$

を手で追える値で用意します。

| 変数 | 1. 意味 | 2. 数式対応 | 3. shape | 4. 軸 | 5. 計算元 | 6. 用途 |
|---|---|---|---|---|---|---|
| `c` | class 数 | $c$ | scalar | なし | 固定値 5 | logits の次元 |
| `l_n` | dense logits | $l_n$ | $(c,)$ | axis 0: class | toy logits | dense prediction と margin の基準 |


In [26]:
c = 5
l_n = torch.tensor(
    [2.4, 1.6, 0.2, -0.5, 1.2],
    dtype=dtype,
    device=device,
)
assert l_n.shape == (c,)


## 3. Classification margin

dense model の予測 class を

$$
y_n^*\coloneqq\operatorname*{argmax}_k(l_n)_k
$$

とします。

最大 competitor の class index を

$$
k_n^{\mathrm{comp}}
\coloneqq
\operatorname*{argmax}_{k\ne y_n^*}(l_n)_k
$$

とします。

classification margin は

$$
\boxed{
\gamma_n
=
(l_n)_{y_n^*}
-
(l_n)_{k_n^{\mathrm{comp}}}
}
$$

であり、

$$
(l_n)_{k_n^{\mathrm{comp}}}
=
\max_{k\ne y_n^*}(l_n)_k
$$

です。

つまり classification margin は **1位と2位の logit 差**です。$\gamma_n$ が大きいほど、順位逆転にはより大きな perturbation が必要になります。

| 変数 | 1. 意味 | 2. 数式対応 | 3. shape | 4. 軸 | 5. 計算元 | 6. 用途 |
|---|---|---|---|---|---|---|
| `y_star` | dense 予測 class | $y_n^*$ | scalar integer | なし | `l_n` の argmax | top class を特定 |
| `top_logit` | top-class logit | $(l_n)_{y_n^*}$ | scalar | なし | `l_n[y_star]` | margin の上側 |
| `competitor_index` | 最大 competitor の class index | $k_n^{\mathrm{comp}}$ | scalar integer | なし | top class を除く logits の argmax | worst-case perturbation の対象 class を特定 |
| `competitor_logit` | 最大 competitor logit | $(l_n)_{k_n^{\mathrm{comp}}}$ | scalar | なし | `l_n[competitor_index]` | 最も危険な competitor の値 |
| `gamma_n` | classification margin | $(l_n)_{y_n^*}-(l_n)_{k_n^{\mathrm{comp}}}$ | scalar | なし | `top_logit - competitor_logit` | prediction の余裕を測る |


In [27]:
# TODO 1:
# y_star, top_logit, competitor_index, competitor_logit, gamma_n
# を計算してください。
from torch import argmax
y_star = argmax(l_n)
top_logit = l_n[y_star]

masked = l_n.clone()
# 値を人為的に変える
masked[y_star] = float("-inf")   # 1位を除外
# 1位を除外したmaskedで2位を求める
competitor_index = torch.argmax(masked)
competitor_logit = l_n[competitor_index]
gamma_n = top_logit - competitor_logit  # 1位 − 2位

### ヒント

- `torch.argmax` で top class index を取得できます。
- 最大 competitor については、top class を除いて最大値とその index を求めます。
- `torch.topk` で上位2要素を取得する方法でも構いません。
- margin は「1位 − 2位」です。


### 検証

TODO 1 を埋めた後に実行してください。


In [28]:
assert y_star.ndim == 0
assert top_logit.ndim == 0
assert competitor_index.ndim == 0
assert competitor_logit.ndim == 0
assert gamma_n.ndim == 0

assert 0 <= int(y_star) < c
assert 0 <= int(competitor_index) < c
assert int(competitor_index) != int(y_star)

assert_close(top_logit, l_n[y_star], rtol=rtol, atol=atol)
assert_close(
    competitor_logit,
    l_n[competitor_index],
    rtol=rtol,
    atol=atol,
)
assert top_logit > competitor_logit
assert gamma_n > 0.0
assert_close(
    gamma_n,
    top_logit - competitor_logit,
    rtol=rtol,
    atol=atol,
)

print(f"predicted class  = {int(y_star)}")
print(f"competitor class = {int(competitor_index)}")
print(f"top logit        = {top_logit.item():.6f}")
print(f"competitor       = {competitor_logit.item():.6f}")
print(f"margin gamma_n   = {gamma_n.item():.6f}")


predicted class  = 0
competitor class = 1
top logit        = 2.400000
competitor       = 1.600000
margin gamma_n   = 0.800000


## 4. Logits perturbation と $\ell_\infty$ norm

compressed logits を

$$
\widetilde l_n=l_n+\delta l_n
$$

とします。

$$
\varepsilon_n=\|\delta l_n\|_\infty=\max_k |(\delta l_n)_k|
$$

と置くと、全 class について

$$
|(\delta l_n)_k|\le\varepsilon_n
$$

が同時に分かります。これが $\ell_\infty$ norm を使う利点です。

| 変数 | 1. 意味 | 2. 数式対応 | 3. shape | 4. 軸 | 5. 計算元 | 6. 用途 |
|---|---|---|---|---|---|---|
| `delta_l_safe` | 安全条件を満たす toy error | $\delta l_n$ | $(c,)$ | axis 0: class | 手で与える値 | stability 条件を確認 |
| `l_tilde_safe` | perturbation 後 logits | $\widetilde l_n$ | $(c,)$ | axis 0: class | `l_n + delta_l_safe` | compressed logits の toy |
| `epsilon_safe` | 最大 logits error | $\|\delta l_n\|_\infty$ | scalar | なし | `delta_l_safe` | $\gamma_n/2$ と比較 |
| `y_tilde_safe` | perturbation 後 prediction | $\arg\max\widetilde l_n$ | scalar integer | なし | `l_tilde_safe` | argmax 不変性を確認 |


In [29]:
delta_l_safe = torch.tensor(
    [-0.20, 0.15, -0.10, 0.05, 0.10],
    dtype=dtype,
    device=device,
)
assert delta_l_safe.shape == (c,)

# TODO 2:
# l_tilde_safe, epsilon_safe, y_tilde_safe を計算し、
l_tilde_safe = l_n + delta_l_safe
epsilon_safe = torch.max(torch.abs(delta_l_safe))
y_tilde_safe = torch.argmax(l_tilde_safe)
# epsilon_safe < gamma_n / 2 と argmax 不変を確認してください。
assert epsilon_safe < gamma_n / 2
assert int(y_tilde_safe) == int(y_star)


### ヒント
- $\ell_\infty$ norm は絶対値の最大値です。
- `torch.linalg.vector_norm(..., ord=float("inf"))` も使えます。
- 比較する条件は $\varepsilon_n<\gamma_n/2$ です。


### 検証

TODO 2 を埋めた後に実行してください。


In [30]:
assert l_tilde_safe.shape == (c,)
assert epsilon_safe.ndim == 0
assert y_tilde_safe.ndim == 0
assert_close(l_tilde_safe, l_n + delta_l_safe, rtol=rtol, atol=atol)
assert epsilon_safe < gamma_n / 2
assert int(y_tilde_safe) == int(y_star)


## 5. Safe case を全 competitor に対して確認する

任意の competitor $k\ne y_n^*$ に対して、

$$
(\widetilde l_n)_{y_n^*}
-
(\widetilde l_n)_k
\ge
\gamma_n-2\varepsilon_n
$$

が成り立つことを、前節の safe perturbation で確認します。

| 変数 | 1. 意味 | 2. 数式対応 | 3. shape | 4. 軸 | 5. 計算元 | 6. 用途 |
|---|---|---|---|---|---|---|
| `k` | 現在確認している class index | $k$ | scalar integer | なし | `range(c)` | 全 competitor を順番に確認 |
| `compressed_gap` | perturbation 後の top-vs-class-$k$ gap | $(\widetilde l_n)_{y_n^*}-(\widetilde l_n)_k$ | scalar | なし | `l_tilde_safe` | 各 competitor に対する実際の差を測る |
| `lower_bound` | 全 competitor に共通する gap 下界 | $\gamma_n-2\varepsilon_n$ | scalar | なし | `gamma_n`, `epsilon_safe` | 理論下界と実測 gap を比較 |


In [31]:
# TODO 3:
# 全 k != y_star について、
# compressed_gap >= gamma_n - 2 * epsilon_safe
# かつ compressed_gap > 0
# を確認してください。
lower_bound = gamma_n - 2 * epsilon_safe

for k in range(c):
    if k == int(y_star):
        continue
    compressed_gap = l_tilde_safe[y_star] - l_tilde_safe[k]
    assert compressed_gap >= lower_bound
    assert compressed_gap > 0


### ヒント

- `for k in range(c)` で全 class を走査できます。
- `k == int(y_star)` のときだけスキップします。
- `lower_bound` は全 competitor に共通です。


### 検証

TODO 3 の中で、各 $k\ne y_n^*$ に対して

$$
\texttt{compressed\_gap}\ge\texttt{lower\_bound}
$$

および

$$
\texttt{compressed\_gap}>0
$$

を `assert` してください。


## 6. なぜ $2\varepsilon_n$ なのか

top class は最悪の場合

$$
(l_n)_{y_n^*}\to(l_n)_{y_n^*}-\varepsilon_n
$$

まで下がり、competitor は

$$
(l_n)_k\to(l_n)_k+\varepsilon_n
$$

まで上がります。

したがって差は

$$
(l_n)_{y_n^*}-(l_n)_k-2\varepsilon_n
$$

まで縮み得ます。

最も強い competitor に対しても正の差を保つには

$$
\gamma_n-2\varepsilon_n>0
$$

すなわち

$$
\boxed{\varepsilon_n<\frac{\gamma_n}{2}}
$$

で十分です。

等号だと worst case で tie が起こり得るため、保証には strict inequality を使います。


### Worst-case perturbation を数値確認する

| 変数 | 1. 意味 | 2. 数式対応 | 3. shape | 4. 軸 | 5. 計算元 | 6. 用途 |
|---|---|---|---|---|---|---|
| `epsilon_worst` | worst-case error の大きさ | $\varepsilon$ | scalar | なし | 手で選ぶ値 | margin 縮小量の確認 |
| `delta_l_worst` | top を下げ competitor を上げる誤差 | $\delta l_n$ | $(c,)$ | axis 0: class | top / competitor index と $\varepsilon$ | 最悪方向を作る |
| `gap_before` | perturbation 前の top-vs-competitor 差 | $\gamma_n$ | scalar | なし | dense logits | 元の margin |
| `gap_after` | perturbation 後の同じ2 class の差 | perturbed top − competitor | scalar | なし | perturbed logits | $2\varepsilon$ 縮小を確認 |


In [32]:
epsilon_worst = torch.tensor(
    0.30,
    dtype=dtype,
    device=device,
)

# TODO 4:
# top class に -epsilon_worst、最大 competitor に +epsilon_worst を与える
# delta_l_worst を作り、gap_before と gap_after を計算してください。
delta_l_worst = torch.zeros(
    c,
    dtype=dtype,
    device=device,
)
delta_l_worst[y_star] = delta_l_worst[y_star] - epsilon_worst
delta_l_worst[competitor_index] = delta_l_worst[competitor_index] + epsilon_worst
gap_before = l_n[y_star] - l_n[competitor_index]
# gap_after は摂動後の logits での差
l_tilde_worst = l_n + delta_l_worst
gap_after = l_tilde_worst[y_star] - l_tilde_worst[competitor_index]

# competitor_index は TODO 1 で求めたものを使います。


### ヒント

- `competitor_index` は TODO 1 で求めた最大 competitor の class index です。
- まずゼロ vector を作り、top class と competitor の2要素だけ変更すると追いやすいです。
- `epsilon_worst` は `dtype` / `device` を揃えた scalar Tensor です。
- 理論上は $\text{gap_after}=\text{gap_before}-2\varepsilon$ です。


### 検証

TODO 4 を埋めた後に実行してください。


In [33]:
assert epsilon_worst.ndim == 0
assert epsilon_worst.dtype == dtype
assert epsilon_worst.device == device

assert delta_l_worst.shape == (c,)
assert gap_before.ndim == 0
assert gap_after.ndim == 0

assert_close(gap_before, gamma_n, rtol=rtol, atol=atol)
assert_close(
    gap_after,
    gap_before - 2.0 * epsilon_worst,
    rtol=rtol,
    atol=atol,
)
assert gap_after > 0.0

print(f"epsilon_worst = {epsilon_worst.item():.6f}")
print(f"gap before    = {gap_before.item():.6f}")
print(f"2 * epsilon   = {(2.0 * epsilon_worst).item():.6f}")
print(f"gap after     = {gap_after.item():.6f}")


epsilon_worst = 0.300000
gap before    = 0.800000
2 * epsilon   = 0.600000
gap after     = 0.200000


## 7. 十分条件であって必要条件ではない

$$
\|\delta l_n\|_\infty<\frac{\gamma_n}{2}
$$

は **十分条件**です。

条件を破ったからといって、必ず argmax が変わるわけではありません。

たとえば全 class に同じ値 $a$ を加えると、$\ell_\infty$ norm は大きくできますが順位は変わりません。

| 変数 | 1. 意味 | 2. 数式対応 | 3. shape | 4. 軸 | 5. 計算元 | 6. 用途 |
|---|---|---|---|---|---|---|
| `delta_l_large_same_shift` | 全 class 共通 shift | $a\mathbf1$ | $(c,)$ | axis 0: class | 定数 vector | 必要条件ではない例 |
| `l_tilde_large_same_shift` | shift 後 logits | $l_n+\delta l_n$ | $(c,)$ | axis 0: class | dense logits + shift | 順位比較 |
| `epsilon_large_same_shift` | shift の $\ell_\infty$ norm | $\|\delta l_n\|_\infty$ | scalar | なし | shift vector | margin 条件との比較 |
| `y_tilde_large_same_shift` | shift 後 prediction | $\arg\max\widetilde l_n$ | scalar integer | なし | shifted logits | argmax 不変を確認 |


In [34]:
delta_l_large_same_shift = torch.ones(
    c,
    dtype=dtype,
    device=device,
)

# TODO 5:
# l_tilde_large_same_shift, epsilon_large_same_shift,
# y_tilde_large_same_shift を計算してください。
# 1. 共通 shift を加えた logits を作る
l_tilde_large_same_shift = l_n + delta_l_large_same_shift

# 2. 誤差ベクトルの infinity norm を計算する
epsilon_large_same_shift = torch.max(torch.abs(delta_l_large_same_shift))

# 3. shift 後 logits の predicted class を計算する
y_tilde_large_same_shift = torch.argmax(l_tilde_large_same_shift)

# 条件は満たさないが argmax は変わらないことを確認します。
assert epsilon_large_same_shift >= gamma_n / 2
assert int(y_tilde_large_same_shift) == int(y_star)


### ヒント
同じ定数 $a$ を全 class に加えても、

$$
(l_n)_i>(l_n)_j\Longrightarrow(l_n)_i+a>(l_n)_j+a
$$

なので順位は変わりません。


### 検証

TODO 5 を埋めた後に実行してください。


In [35]:
assert l_tilde_large_same_shift.shape == (c,)
assert epsilon_large_same_shift.ndim == 0
assert y_tilde_large_same_shift.ndim == 0
assert epsilon_large_same_shift >= gamma_n / 2
assert int(y_tilde_large_same_shift) == int(y_star)


## 8. 一般形の導出

任意の competitor $k\ne y_n^*$ に対して、

$$
\begin{aligned}
(\widetilde l_n)_{y_n^*}-(\widetilde l_n)_k
&=(l_n)_{y_n^*}+(\delta l_n)_{y_n^*}-(l_n)_k-(\delta l_n)_k\\
&\ge (l_n)_{y_n^*}-(l_n)_k-2\varepsilon_n\\
&\ge \gamma_n-2\varepsilon_n.
\end{aligned}
$$

したがって

$$
\varepsilon_n<\frac{\gamma_n}{2}
$$

なら全 competitor に対して差が正なので、

$$
\boxed{
\|\delta l_n\|_\infty<\frac{\gamma_n}{2}
\Longrightarrow
\operatorname*{argmax}_k(\widetilde l_n)_k
=\operatorname*{argmax}_k(l_n)_k
}
$$

です。


## 9. まとめ

今回の中心は次の3点です。

### Classification margin

$$
\gamma_n=(l_n)_{y_n^*}-\max_{k\ne y_n^*}(l_n)_k
$$

### Logits error

$$
\varepsilon_n=\|\delta l_n\|_\infty
$$

### Argmax stability の十分条件

$$
\boxed{\|\delta l_n\|_\infty<\frac{\gamma_n}{2}}
$$

なら prediction は変わりません。

理由は、top class が最大 $\varepsilon_n$ 下がり、competitor が最大 $\varepsilon_n$ 上がるため、margin が最大 $2\varepsilon_n$ 縮み得るからです。

これは十分条件であって必要条件ではありません。

次は

$$
\|\delta l_n\|_\infty\le\|\delta l_n\|_2\le\|\Delta L\|_F
$$

を使って、03 の logits error upper bound と margin を接続できますが、この Notebook ではまだ進みません。
